# PROMISE12 Colab Notebook

This notebook first runs a 2-case smoke test (Case00 and Case01). After that, set `SMOKE_TEST = False` to run Case00-Case24.


In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
import textwrap
import traceback
from pathlib import Path

ROOT = "/content"
RAW_DIR = "/content/raw_promise12"
REPO_DIR = "/content/MRI-Prostate-Gland-and-Zone-Segmentor"
PATS_DIR = "/content/MRI-Prostate-Gland-and-Zone-Segmentor/Pats"
OUTPUTS_DIR = "/content/MRI-Prostate-Gland-and-Zone-Segmentor/Outputs"
DICOM_OUTPUTS_DIR = "/content/MRI-Prostate-Gland-and-Zone-Segmentor/dicom_outputs"

ROOT_PATH = Path(ROOT)
RAW_PATH = Path(RAW_DIR)
REPO_PATH = Path(REPO_DIR)
PATS_PATH = Path(PATS_DIR)
OUTPUTS_PATH = Path(OUTPUTS_DIR)
DICOM_OUTPUTS_PATH = Path(DICOM_OUTPUTS_DIR)

SMOKE_TEST = True
SELECTED_CASES = ["Case00", "Case01"] if SMOKE_TEST else [f"Case{i:02d}" for i in range(25)]

def run_cmd(cmd, cwd=None, env=None, check=True):
    """Run a command, print stdout/stderr, and fail loudly if requested."""
    printable = cmd if isinstance(cmd, str) else " ".join(map(str, cmd))
    print(f"\n$ {printable}")
    completed = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {completed.returncode}: {printable}"
        )
    return completed

def print_tree(root):
    root = Path(root)
    if not root.exists():
        print(f"Missing tree root: {root}")
        return
    print(f"\nTree under {root}:")
    print(root)
    for path in sorted(root.rglob("*")):
        rel = path.relative_to(root)
        prefix = "[D]" if path.is_dir() else "[F]"
        print(f"{prefix} {rel}")

print(f"ROOT: {ROOT}")
print(f"RAW_DIR: {RAW_DIR}")
print(f"REPO_DIR: {REPO_DIR}")
print(f"PATS_DIR: {PATS_DIR}")
print(f"OUTPUTS_DIR: {OUTPUTS_DIR}")
print(f"DICOM_OUTPUTS_DIR: {DICOM_OUTPUTS_DIR}")
print(f"SMOKE_TEST: {SMOKE_TEST}")
print(f"Selected cases: {SELECTED_CASES}")


In [ ]:
import importlib.metadata as md

if REPO_PATH.exists():
    print(f"Removing existing repo at {REPO_PATH}")
    shutil.rmtree(REPO_PATH)

run_cmd(["apt-get", "update"])
run_cmd(["apt-get", "install", "-y", "git-lfs"])
run_cmd(["git", "lfs", "install"])

clone_env = os.environ.copy()
clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
run_cmd(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/dzaridis/MRI-Prostate-Gland-and-Zone-Segmentor.git",
        REPO_DIR,
    ],
    env=clone_env,
)

lfs_include = ",".join(
    [
        "nnUnet_paths/nnUNet_results/Dataset016_WgSegmentationPNetAndPicai/**",
        "nnUnet_paths/nnUNet_results/Dataset019_ProstateZonesSegmentationWgFilteredLessDilated/**",
    ]
)
run_cmd(["git", "lfs", "pull", "--include", lfs_include], cwd=REPO_DIR)

run_cmd([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "pip"])
run_cmd([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_PATH / "requirements.txt")])
run_cmd(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "SimpleITK",
        "nibabel",
        "numpy",
        "pandas",
        "matplotlib",
    ]
)

required_final_checkpoints = [
    REPO_PATH
    / "nnUnet_paths/nnUNet_results/Dataset016_WgSegmentationPNetAndPicai/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/checkpoint_final.pth",
    REPO_PATH
    / "nnUnet_paths/nnUNet_results/Dataset019_ProstateZonesSegmentationWgFilteredLessDilated/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/checkpoint_final.pth",
]

repaired = []
for final_ckpt in required_final_checkpoints:
    best_ckpt = final_ckpt.with_name("checkpoint_best.pth")
    if not final_ckpt.exists() and best_ckpt.exists():
        shutil.copy2(best_ckpt, final_ckpt)
        repaired.append((best_ckpt, final_ckpt))

if repaired:
    print("Copied checkpoint_best.pth to checkpoint_final.pth for:")
    for src_path, dst_path in repaired:
        print(f"  {src_path} -> {dst_path}")

missing_checkpoints = [str(path) for path in required_final_checkpoints if not path.exists()]
if missing_checkpoints:
    raise FileNotFoundError(
        "Missing model checkpoints after git lfs pull. "
        "Check Git LFS access and rerun the cell.\n"
        + "\n".join(missing_checkpoints)
    )

def safe_version(package_name):
    try:
        return md.version(package_name)
    except md.PackageNotFoundError:
        return "not installed"

print(f"Python: {sys.version}")
print(f"torch: {safe_version('torch')}")
print(f"nnunetv2: {safe_version('nnunetv2')}")
print(f"SimpleITK: {safe_version('SimpleITK')}")
print(f"nibabel: {safe_version('nibabel')}")
print(f"numpy: {safe_version('numpy')}")
print(f"pandas: {safe_version('pandas')}")
print(f"MedProIO: {safe_version('MedProIO')}")
run_cmd(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)


In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"RAW_DIR does not exist: {RAW_PATH}. Upload your PROMISE12 .nii files to /content/raw_promise12 first."
    )

print(f"Listing files under {RAW_PATH}:")
raw_entries = sorted(RAW_PATH.iterdir())
for entry in raw_entries:
    kind = "DIR " if entry.is_dir() else "FILE"
    print(f"  {kind}: {entry.name}")

image_pattern = re.compile(r"^(Case\d{2})\.nii$")
gt_pattern = re.compile(r"^(Case\d{2})_segmentation\.nii$")

image_cases = {}
gt_cases = {}
unexpected_files = []

for entry in raw_entries:
    if not entry.is_file():
        unexpected_files.append(entry.name)
        continue

    image_match = image_pattern.match(entry.name)
    gt_match = gt_pattern.match(entry.name)

    if image_match:
        image_cases[image_match.group(1)] = entry
    elif gt_match:
        gt_cases[gt_match.group(1)] = entry
    else:
        unexpected_files.append(entry.name)

all_case_ids = sorted(set(image_cases) | set(gt_cases))
paired_cases = sorted(set(image_cases) & set(gt_cases))
missing_image_cases = sorted(set(gt_cases) - set(image_cases))
missing_gt_cases = sorted(set(image_cases) - set(gt_cases))

print(f"\nDetected case ids: {all_case_ids}")
print(f"Paired cases: {paired_cases}")
if missing_image_cases:
    print(f"Cases missing image files: {missing_image_cases}")
if missing_gt_cases:
    print(f"Cases missing GT files: {missing_gt_cases}")
if unexpected_files:
    print(f"Unexpected files or directories: {unexpected_files}")

if not paired_cases:
    raise RuntimeError(
        "No valid CaseXX.nii / CaseXX_segmentation.nii pairs were found in RAW_DIR."
    )

missing_selected_images = [case_id for case_id in SELECTED_CASES if case_id not in image_cases]
missing_selected_gts = [case_id for case_id in SELECTED_CASES if case_id not in gt_cases]

if missing_selected_images or missing_selected_gts:
    raise FileNotFoundError(
        "Required cases for the current run are missing.\n"
        f"Missing images: {missing_selected_images}\n"
        f"Missing GT masks: {missing_selected_gts}\n"
        f"Selected cases: {SELECTED_CASES}"
    )

CASE_IMAGE_PATHS = {case_id: image_cases[case_id] for case_id in paired_cases}
CASE_GT_PATHS = {case_id: gt_cases[case_id] for case_id in paired_cases}
print("\nValidation complete. Selected cases are present.")


In [ ]:
for folder in [PATS_PATH, OUTPUTS_PATH, DICOM_OUTPUTS_PATH]:
    if not str(folder).startswith(REPO_DIR):
        raise RuntimeError(f"Refusing to modify a path outside the repo folder: {folder}")

    if folder.exists():
        print(f"Removing existing folder: {folder}")
        shutil.rmtree(folder)

    folder.mkdir(parents=True, exist_ok=True)
    print(f"Created clean folder: {folder}")


In [ ]:
import SimpleITK as sitk

CONVERTED_INPUTS = {}

for case_id in SELECTED_CASES:
    src_path = CASE_IMAGE_PATHS[case_id]
    dst_path = PATS_PATH / f"{case_id}.nii.gz"
    image = sitk.ReadImage(str(src_path))
    sitk.WriteImage(image, str(dst_path), useCompression=True)
    CONVERTED_INPUTS[case_id] = dst_path
    print(f"Converted {src_path} -> {dst_path}")

segmentation_files_in_pats = list(PATS_PATH.glob("*segmentation*"))
if segmentation_files_in_pats:
    raise RuntimeError(
        "Ground-truth segmentation files were found inside Pats, which must contain input MR images only.\n"
        + "\n".join(str(path) for path in segmentation_files_in_pats)
    )

print("\nPats contents:")
for path in sorted(PATS_PATH.iterdir()):
    print(f"  {path.name}")


In [ ]:
import numpy as np
import SimpleITK as sitk

for case_id in SELECTED_CASES:
    print("\n" + "=" * 90)
    print(f"Case: {case_id}")

    input_img = sitk.ReadImage(str(CASE_IMAGE_PATHS[case_id]))
    gt_img = sitk.ReadImage(str(CASE_GT_PATHS[case_id]))

    input_arr = sitk.GetArrayFromImage(input_img)
    gt_arr = sitk.GetArrayFromImage(gt_img)
    gt_unique = np.unique(gt_arr)

    print(f"Input image path: {CASE_IMAGE_PATHS[case_id]}")
    print(f"GT mask path:    {CASE_GT_PATHS[case_id]}")
    print(f"Input shape (z, y, x): {input_arr.shape}")
    print(f"Input spacing: {input_img.GetSpacing()}")
    print(f"Input origin:  {input_img.GetOrigin()}")
    print(f"Input direction: {input_img.GetDirection()}")
    print(f"GT shape (z, y, x): {gt_arr.shape}")
    print(f"GT spacing: {gt_img.GetSpacing()}")
    print(f"GT origin:  {gt_img.GetOrigin()}")
    print(f"GT direction: {gt_img.GetDirection()}")
    print(f"GT unique values (up to 20 shown): {gt_unique[:20].tolist()}")

    if set(gt_unique.astype(float).tolist()) <= {0.0, 1.0}:
        print("GT is already binary for whole-gland evaluation.")
    else:
        print("GT has values beyond {0, 1}; evaluation will binarize with gt > 0.")


In [ ]:
def print_matching_lines(file_path, keywords):
    print(f"\n--- {file_path.relative_to(REPO_PATH)} ---")
    text = file_path.read_text(encoding="utf-8")
    for lineno, line in enumerate(text.splitlines(), start=1):
        if any(keyword in line for keyword in keywords):
            print(f"{lineno:4d}: {line}")
    return text

main_text = print_matching_lines(
    REPO_PATH / "__main__.py",
    ["INPUT_VOLUME", "OUTPUT_VOLUME", "converter()", "upload(", "dicom_outputs"],
)
get_images_text = print_matching_lines(
    REPO_PATH / "Utils/get_images.py",
    [".nii.gz", ".dcm", "No .nii.gz or .dcm file was found", "patient_dict.yaml"],
)
helpers_text = print_matching_lines(
    REPO_PATH / "Utils/helpers.py",
    ["ResampledToOriginalSegmentationPaths.json", "nnOutputSegmentationPaths.json", "wg_binary"],
)

if 'INPUT_VOLUME = "Pats"' not in main_text or 'OUTPUT_VOLUME = "Outputs"' not in main_text:
    raise RuntimeError("__main__.py no longer points to Pats / Outputs as expected.")

if 'if file.endswith(".nii.gz")' not in get_images_text:
    raise RuntimeError("Utils/get_images.py no longer scans .nii.gz files as expected.")

if "ResampledToOriginalSegmentationPaths.json" not in helpers_text or "wg_binary" not in helpers_text:
    raise RuntimeError("The expected output JSON / wg_binary references were not found.")

print("\nNotebook execution plan:")
print("- Keep the .nii -> .nii.gz conversion step because the repo loader scans Pats for .nii.gz.")
print("- Run the segmentation core from the repo root and skip converter()/upload() because Colab evaluation only needs NIfTI outputs.")
print("- Evaluate the resampled whole-gland output referenced by ResampledToOriginalSegmentationPaths.json when available.")


In [ ]:
def build_inference_script():
    return textwrap.dedent(
        f'''
        import os
        import sys
        import traceback

        repo_dir = {REPO_DIR!r}
        os.chdir(repo_dir)
        sys.path.insert(0, repo_dir)

        print(f"Working directory: {{os.getcwd()}}")
        print("Running segmentation core only (no DICOM conversion / no Orthanc upload).")

        try:
            from Utils.get_images import get_images
            from Utils import InputCheck, helpers, segmentor_pipeline

            patient_list = get_images("Pats")
            print(f"patient_list: {{patient_list}}")
            pats = InputCheck.load_nii_gz_files(patient_list)
            print(f"Loaded patient keys: {{list(pats.keys())}}")

            segmentor_pipeline.segmentor_pipeline_operation(
                output_volume="Outputs",
                pats=pats,
            )
            helpers.process_masks(out_volume="Outputs")
            print("Segmentation pipeline completed successfully.")
        except Exception:
            traceback.print_exc()
            raise
        '''
    )

inference_cmd = [sys.executable, "-c", build_inference_script()]
inference_env = os.environ.copy()
inference_env["PYTHONUNBUFFERED"] = "1"

first_attempt = run_cmd(inference_cmd, cwd=REPO_DIR, env=inference_env, check=False)

if first_attempt.returncode != 0:
    combined_log = (first_attempt.stdout or "") + "\n" + (first_attempt.stderr or "")
    print("\nFirst inference attempt failed. Combined log:")
    print(combined_log)

    repairs = []
    for final_ckpt in [
        REPO_PATH
        / "nnUnet_paths/nnUNet_results/Dataset016_WgSegmentationPNetAndPicai/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/checkpoint_final.pth",
        REPO_PATH
        / "nnUnet_paths/nnUNet_results/Dataset019_ProstateZonesSegmentationWgFilteredLessDilated/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/checkpoint_final.pth",
    ]:
        best_ckpt = final_ckpt.with_name("checkpoint_best.pth")
        if not final_ckpt.exists() and best_ckpt.exists():
            shutil.copy2(best_ckpt, final_ckpt)
            repairs.append(f"Copied {best_ckpt.name} -> {final_ckpt.name} for {final_ckpt.parent}")

    if "No .nii.gz or .dcm file was found" in combined_log:
        print("Diagnosis: Pats does not contain the converted .nii.gz inputs. Re-run Cell 5 before retrying.")
    if "checkpoint_final.pth" in combined_log:
        print("Diagnosis: the repo could not find a required nnU-Net checkpoint.")
    if "ModuleNotFoundError" in combined_log:
        print("Diagnosis: a Python dependency is missing. Re-run Cell 2 and inspect the pip install output.")

    if repairs:
        print("\nApplied smallest safe repair(s):")
        for repair in repairs:
            print(f"- {repair}")
        second_attempt = run_cmd(inference_cmd, cwd=REPO_DIR, env=inference_env, check=False)
    else:
        second_attempt = first_attempt

    if second_attempt.returncode != 0:
        raise RuntimeError(
            "Inference failed after one retry. Inspect the printed traceback / logs above before continuing."
        )

produced_outputs = list(OUTPUTS_PATH.rglob("*.nii.gz")) + list(OUTPUTS_PATH.glob("*.json"))
if not produced_outputs:
    raise RuntimeError(
        f"Inference finished without producing expected output files under {OUTPUTS_PATH}."
    )

print(f"Produced {len(produced_outputs)} output files under {OUTPUTS_PATH}.")


In [ ]:
print_tree(OUTPUTS_PATH)

json_candidates = [
    OUTPUTS_PATH / "ResampledToOriginalSegmentationPaths.json",
    OUTPUTS_PATH / "nnOutputSegmentationPaths.json",
]
OUTPUT_JSONS = {}

for json_path in json_candidates:
    if json_path.exists():
        payload = json.loads(json_path.read_text(encoding="utf-8"))
        OUTPUT_JSONS[json_path.name] = payload
        print(f"\nLoaded {json_path.name} with {len(payload)} top-level entries.")
        case_preview = {case_id: payload.get(case_id, "<missing>") for case_id in SELECTED_CASES}
        print(json.dumps(case_preview, indent=2))
    else:
        print(f"Missing JSON file: {json_path}")

def normalize_repo_output_path(path_string):
    normalized = str(path_string).replace("\\", "/")
    candidate = Path(normalized)
    if candidate.is_absolute():
        return candidate
    return REPO_PATH / candidate

def recursive_find_key(obj, target_key):
    matches = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == target_key and isinstance(value, str):
                matches.append(value)
            matches.extend(recursive_find_key(value, target_key))
    elif isinstance(obj, list):
        for item in obj:
            matches.extend(recursive_find_key(item, target_key))
    return matches

def extract_case_wg_binary(case_id, payload):
    if case_id not in payload:
        return None
    matches = recursive_find_key(payload[case_id], "wg_binary")
    if not matches:
        return None
    if len(matches) > 1:
        print(f"Multiple wg_binary matches found for {case_id}; using the first one: {matches}")
    return normalize_repo_output_path(matches[0])

PREDICTION_PATHS = {}
PREDICTION_SOURCES = {}

for case_id in SELECTED_CASES:
    pred_path = None
    pred_source = None

    if "ResampledToOriginalSegmentationPaths.json" in OUTPUT_JSONS:
        pred_path = extract_case_wg_binary(case_id, OUTPUT_JSONS["ResampledToOriginalSegmentationPaths.json"])
        if pred_path is not None:
            pred_source = "ResampledToOriginalSegmentationPaths.json"

    if pred_path is None and "nnOutputSegmentationPaths.json" in OUTPUT_JSONS:
        pred_path = extract_case_wg_binary(case_id, OUTPUT_JSONS["nnOutputSegmentationPaths.json"])
        if pred_path is not None:
            pred_source = "nnOutputSegmentationPaths.json"

    if pred_path is None:
        fallback_candidates = sorted((OUTPUTS_PATH / case_id).rglob("wg_binary.nii.gz"))
        if fallback_candidates:
            pred_path = fallback_candidates[0]
            pred_source = "filesystem_fallback"
            print(
                f"Warning: JSON lookup failed for {case_id}; using the discovered file-system path {pred_path}"
            )

    if pred_path is None:
        raise FileNotFoundError(
            f"Could not locate a whole-gland prediction for {case_id}. Inspect the output tree printed above."
        )

    if not pred_path.exists():
        raise FileNotFoundError(
            f"Prediction path for {case_id} does not exist: {pred_path}"
        )

    PREDICTION_PATHS[case_id] = pred_path
    PREDICTION_SOURCES[case_id] = pred_source
    print(f"{case_id} -> {pred_path} (source: {pred_source})")


In [ ]:
import numpy as np
import SimpleITK as sitk

def geometry_signature(image):
    return {
        "size_xyz": tuple(int(v) for v in image.GetSize()),
        "spacing_xyz": tuple(round(float(v), 6) for v in image.GetSpacing()),
        "origin_xyz": tuple(round(float(v), 6) for v in image.GetOrigin()),
        "direction": tuple(round(float(v), 6) for v in image.GetDirection()),
    }

def same_geometry(image_a, image_b):
    return geometry_signature(image_a) == geometry_signature(image_b)

EVAL_ITEMS = []

for case_id in SELECTED_CASES:
    pred_path = PREDICTION_PATHS[case_id]
    gt_path = CASE_GT_PATHS[case_id]

    pred_img = sitk.ReadImage(str(pred_path))
    gt_img = sitk.ReadImage(str(gt_path))

    pred_shape_before = sitk.GetArrayFromImage(pred_img).shape
    gt_shape = sitk.GetArrayFromImage(gt_img).shape

    print("\n" + "=" * 90)
    print(f"Case: {case_id}")
    print(f"Prediction source: {PREDICTION_SOURCES[case_id]}")
    print(f"Prediction path: {pred_path}")
    print(f"GT path: {gt_path}")
    print(f"Prediction shape before any resampling (z, y, x): {pred_shape_before}")
    print(f"GT shape (z, y, x): {gt_shape}")

    resampled_to_gt = False
    if not same_geometry(pred_img, gt_img):
        print("Geometry mismatch detected. Resampling prediction to GT geometry with nearest-neighbor interpolation.")
        pred_img = sitk.Resample(
            pred_img,
            gt_img,
            sitk.Transform(),
            sitk.sitkNearestNeighbor,
            0,
            pred_img.GetPixelID(),
        )
        resampled_to_gt = True

    pred_shape_after = sitk.GetArrayFromImage(pred_img).shape
    print(f"Prediction shape after resampling check (z, y, x): {pred_shape_after}")

    EVAL_ITEMS.append(
        {
            "case_id": case_id,
            "image_path": CASE_IMAGE_PATHS[case_id],
            "pred_path": pred_path,
            "gt_path": gt_path,
            "pred_img": pred_img,
            "gt_img": gt_img,
            "resampled_to_gt": resampled_to_gt,
        }
    )


In [ ]:
import numpy as np
import SimpleITK as sitk

def binary_dice(pred_array, gt_array):
    pred_bin = pred_array > 0
    gt_bin = gt_array > 0

    pred_sum = int(pred_bin.sum())
    gt_sum = int(gt_bin.sum())

    if pred_sum == 0 and gt_sum == 0:
        return 1.0
    if pred_sum == 0 or gt_sum == 0:
        return 0.0

    intersection = int(np.logical_and(pred_bin, gt_bin).sum())
    return float(2.0 * intersection / (pred_sum + gt_sum))

results = []
EVAL_BY_CASE = {}

for item in EVAL_ITEMS:
    pred_array = sitk.GetArrayFromImage(item["pred_img"])
    gt_array = sitk.GetArrayFromImage(item["gt_img"])
    dice_value = binary_dice(pred_array, gt_array)

    result = {
        "case_id": item["case_id"],
        "dice": dice_value,
        "pred_path": str(item["pred_path"]),
        "gt_path": str(item["gt_path"]),
        "pred_shape": tuple(pred_array.shape),
        "gt_shape": tuple(gt_array.shape),
    }
    results.append(result)
    EVAL_BY_CASE[item["case_id"]] = {
        **item,
        "pred_array": pred_array,
        "gt_array": gt_array,
    }

    print(f"{item['case_id']}: Dice = {dice_value:.4f}")


In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).sort_values("case_id").reset_index(drop=True)
mean_dice = float(results_df["dice"].mean()) if not results_df.empty else float("nan")

report_stem = "promise12_dice_report_smoketest" if SMOKE_TEST else "promise12_dice_report_first25"
TXT_REPORT_PATH = ROOT_PATH / f"{report_stem}.txt"
CSV_REPORT_PATH = ROOT_PATH / f"{report_stem}.csv"

report_lines = [
    "PROMISE12 Whole-Gland Dice Report",
    f"Evaluated cases: {len(results_df)}",
    "--------------------------------",
]
for row in results_df.itertuples(index=False):
    report_lines.append(f"{row.case_id}: {row.dice:.4f}")
report_lines.extend(
    [
        "--------------------------------",
        f"Mean Dice: {mean_dice:.4f}",
    ]
)

TXT_REPORT_PATH.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

csv_df = results_df.copy()
csv_df["pred_shape"] = csv_df["pred_shape"].apply(str)
csv_df["gt_shape"] = csv_df["gt_shape"].apply(str)
summary_row = {
    "case_id": "MEAN",
    "dice": mean_dice,
    "pred_path": "",
    "gt_path": "",
    "pred_shape": "",
    "gt_shape": "",
}
csv_df = pd.concat([csv_df, pd.DataFrame([summary_row])], ignore_index=True)
csv_df.to_csv(CSV_REPORT_PATH, index=False)

print(TXT_REPORT_PATH.read_text(encoding="utf-8"))
print(f"Saved TXT report to: {TXT_REPORT_PATH}")
print(f"Saved CSV report to: {CSV_REPORT_PATH}")


In [ ]:
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
display(results_df)
print(f"\nMean Dice: {mean_dice:.4f}")
print(f"TXT report: {TXT_REPORT_PATH}")
print(f"CSV report: {CSV_REPORT_PATH}")
print(
    "\nTo switch from the 2-case smoke test to the first 25 cases, set SMOKE_TEST = False in Cell 1 and rerun Cells 1-13."
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import SimpleITK as sitk

VIS_CASE = SELECTED_CASES[0]
case_bundle = EVAL_BY_CASE[VIS_CASE]

image_img = sitk.ReadImage(str(CASE_IMAGE_PATHS[VIS_CASE]))
gt_img = case_bundle["gt_img"]
pred_img = case_bundle["pred_img"]

if not same_geometry(image_img, gt_img):
    print("Resampling input image to GT geometry for visualization.")
    image_img = sitk.Resample(
        image_img,
        gt_img,
        sitk.Transform(),
        sitk.sitkLinear,
        0.0,
        image_img.GetPixelID(),
    )

image_array = sitk.GetArrayFromImage(image_img)
gt_array = sitk.GetArrayFromImage(gt_img) > 0
pred_array = sitk.GetArrayFromImage(pred_img) > 0

candidate_slices = np.where(gt_array.any(axis=(1, 2)) | pred_array.any(axis=(1, 2)))[0]
slice_idx = int(candidate_slices[len(candidate_slices) // 2]) if len(candidate_slices) else image_array.shape[0] // 2
print(f"Visualizing {VIS_CASE} at slice index {slice_idx}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_array[slice_idx], cmap="gray")
axes[0].set_title(f"{VIS_CASE} image")
axes[1].imshow(gt_array[slice_idx], cmap="gray")
axes[1].set_title("GT whole gland")
axes[2].imshow(pred_array[slice_idx], cmap="gray")
axes[2].set_title("Pred whole gland")

for axis in axes:
    axis.axis("off")

plt.tight_layout()
plt.show()
